<font color='red' size='6'>REMINDER</font>

# Fill in any place that says `YOUR CODE HERE`.
- You should remove the line that says `raise NotImplementedError()`. If you do not, your code will (unsurprisingly) throw a run-time error and cause everything to fail.
- Do **NOT** write your answer anywhere else other than where it says `YOUR CODE HERE`. Simply write your code directly below this comment in the **same code cell**.

# Make sure everything runs as expected.
- Go to the menubar, select *Kernel* > *Restart & Run All*

# Do <ins>NOT</ins> change the title (i.e., file name) of this notebook.

# Do <ins>NOT</ins> delete any of the cells in this notebook.

# Make sure you save your work
- Go to the menubar, select *File* > *Save and Checkpoint*

In [1]:
# Run this code cell
from nose.tools import assert_equal, assert_in, assert_is_instance
from nose.tools import assert_almost_equal, assert_not_equal, assert_not_in

# 1. Getting Data

You have received another round of data from Pallet Movers. However, this time a different person from their IT department generated the data and sent it to you because your normal contact was out on PTO. You suspect that the data may not be as clean as the previous file you had. The file is named `suspect_data.xlsx` and resides in the folder `data`. There should be three worksheets in the file.

Read the entire Excel document into a variable named `suspect_data`. 

In [2]:
# import pandas with the alias pd
import pandas as pd

suspect_data = pd.read_excel('./data/suspect_data.xlsx', sheet_name=None)

In [3]:
# This is a test cell
# If **NO** message is printed, it means that the tests passed
# One basic test to see if your code works
assert_equal(type(suspect_data), dict, msg='Your suspect_data variable is not a dictionary')

In [4]:
# This is a test cell
# If **NO** message is printed, it means that the tests passed
# One basic test to see if your code works
assert_equal(len(suspect_data), 3, msg='You did not find the correct number of worksheets')

# 2. Duplicate Hubs?

Are there any hubs duplicated in the first worksheet of the data file? In this case, a duplicate is defined by the hubs having the same city, state, and address. 

1. Find all the hubs that are duplicates, keeping and storing **all** of them in a `DataFrame` named `hub_dupes`.
2. You have decided to drop all the duplicates **except** the first one. Create a `DataFrame` named `hubs` that contains all of the unique hubs.

In [5]:
hub_dupes = suspect_data['hubs'][suspect_data['hubs'].duplicated(subset=['city', 'state', 'address'], keep=False)]

hubs = suspect_data['hubs'].drop_duplicates(subset=['city', 'state', 'address'])
hubs

,id,name,city,state,zip,address
0,Hub-0001,PIT,Pittsburgh,PA,15106,1 North Bell Avenue
1,Hub-0002,NEW,Newark,NJ,7105,500 Doremus Avenue
2,Hub-0003,RIC,Richmond,VA,23224,1636 Commerce Road
3,Hub-0004,CLT,Charlotte,NC,28269,3301 Rotary Drive
4,Hub-0005,ATL,Atlanta,GA,30318,3105 Donald Lee Hollowell Parkway NW


In [6]:
# This is a test cell
# If **NO** message is printed, it means that the tests passed
# Did you correctly name the variables?
assert_in('hub_dupes', dir(), msg="You did not name your variable `hub_dupes` as instructed.")
assert_in('hubs', dir(), msg="You did not name your variable `hubs` as instructed.")

<hr>

<hr>

# 3. Duplicate Spokes?

Are there any spokes duplicated in the second worksheet of the data file? In this case, a duplicate is defined by the spokes having the same city, state, and address. 

1. Find all the spokes that are duplicates, keeping and storing **all** of them in a `DataFrame` named `spoke_dupes`.
2. You have decided to drop the spoke duplicates that do not have an associated hub in the same state as as the spoke. That is, if you have two spokes that are duplicates in the state of South Carolina, but one points to a hub in Virginia and the other points to a hub in South Carolina, then you want to keep the one pointing to the hub in South Carolina and get rid of the one pointing to the hub in Virginia. Create a `DataFrame` named `spokes` that contains all of the unique spokes.

In [7]:
spoke_dupes = suspect_data['spokes'][suspect_data['spokes'].duplicated(subset=['city', 'state', 'address'], keep=False)]

dupe_check = spoke_dupes.merge(hubs, left_on='hub_id', right_on='id', how='left')
dupe_check = dupe_check[dupe_check['state_x'] != dupe_check['state_y']]

spokes = suspect_data['spokes'].drop(suspect_data['spokes'][suspect_data['spokes']['id'].isin(dupe_check['id_x'])].index)
spokes


,id,name,hub_id,city,state,zip,address
0,Spoke-0001,AKR,Hub-0001,Akron,OH,44306,1250 Kelly Avenue
1,Spoke-0002,YNG,Hub-0001,Youngstown,OH,44506,799 Wilson Avenue
2,Spoke-0003,ALN,Hub-0001,Allentown,PA,18103,2229 Lehigh Street
3,Spoke-0004,HAR,Hub-0001,Harrisburg,PA,17110,3900 Industrial Road
4,Spoke-0005,MGN,Hub-0001,Morgantown,WV,26505,89 Brookhaven Road
5,Spoke-0006,WHP,Hub-0002,White Plains,NY,10603,66 Grant Avenue
6,Spoke-0007,TRN,Hub-0002,Trenton,NJ,8608,327 South Broad Street
7,Spoke-0008,PHI,Hub-0002,Philadelphia,PA,19148,48 Snyder Avenue
8,Spoke-0009,ELK,Hub-0002,Elkton,MD,21921,304 East Pulaski Highway
9,Spoke-0010,CAT,Hub-0002,Catskill,NY,12414,17 Maple Avenue


In [8]:
# This is a test cell
# If **NO** message is printed, it means that the tests passed
# Did you correctly name the variables?
assert_in('spoke_dupes', dir(), msg="You did not name your variable `spoke_dupes` as instructed.")
assert_in('spokes', dir(), msg="You did not name your variable `spokes` as instructed.")

<hr>

<hr>

<hr>

# 4. Duplicate Requests?

Are there any requests duplicated in the third worksheet of the data file? In this case, a duplicate is defined by the requests having the same id. 

1. Find all the requests that are duplicates, keeping and storing **all** of them in a `DataFrame` named `request_dupes`.
2. If the duplicates are exactly the same across all fields/columns, then you can simply delete one of the rows. Create a `DataFrame` named `requests_round_one` that results from the task of dropping all but one of the duplicates where all the fields/columns are the same.
3. Your variable `requests_round_one` still contains duplicates based off the id. These duplicates differ on only one or a few different columns. Therefore, you must be more careful in how you decide which rows to keep and which to remove. Handle the remaining duplicates in the following manner, storing your results in a variable named `requests`.
- If the handling unit is a box, the weight should not be more than a ton; drop any duplicate rows that have a handling unit of a box and weigh more than a ton.
- If the handling unit is a crate or pallet, keep the row that has the **larger** weight.

In [9]:
request_dupes = suspect_data['requests'][suspect_data['requests'].duplicated(subset='request_id', keep=False)]
requests_round_one = suspect_data['requests'].drop_duplicates()


requests = requests_round_one[~((requests_round_one['handling_unit'] == 'box') &
         (requests_round_one['weight'] > 2000))].sort_values('weight', ascending=False).drop_duplicates(subset='request_id')
requests.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1170 entries, 133 to 612
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   request_id                    1170 non-null   object 
 1   origin_city                   1170 non-null   object 
 2   origin_state                  1170 non-null   object 
 3   origin_liftgate_service       1170 non-null   int64  
 4   destination_city              1170 non-null   object 
 5   destination_state             1170 non-null   object 
 6   destination_liftgate_service  1170 non-null   int64  
 7   handling_unit                 1170 non-null   object 
 8   length                        1170 non-null   int64  
 9   width                         1170 non-null   int64  
 10  height                        1170 non-null   int64  
 11  weight                        1166 non-null   float64
dtypes: float64(1), int64(5), object(6)
memory usage: 118.8+ KB


<hr>

<hr>

<hr>

<hr>

<hr>

<hr>

# 5. Missing Data?

Are there any requests that have missing data? Using your newly created de-duplicated variable `requests` determine if there are any rows that have missing data. 

1. Find all the requests that have missing and store them in a `DataFrame` named `requests_with_missing_data`.
2. If the missing data is in the `weight` column, then you want to fill it in with the **median of rows containing the same handling unit, length, width, and height**.

In [10]:
requests_with_missing_data = requests[requests.isna().any(axis=1)]
requests_with_missing_data

median_weights = requests.groupby(['handling_unit', 'length', 'width', 'height'])['weight'].transform('median')

requests['weight'] = requests['weight'].fillna(median_weights)
requests.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1170 entries, 133 to 612
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   request_id                    1170 non-null   object 
 1   origin_city                   1170 non-null   object 
 2   origin_state                  1170 non-null   object 
 3   origin_liftgate_service       1170 non-null   int64  
 4   destination_city              1170 non-null   object 
 5   destination_state             1170 non-null   object 
 6   destination_liftgate_service  1170 non-null   int64  
 7   handling_unit                 1170 non-null   object 
 8   length                        1170 non-null   int64  
 9   width                         1170 non-null   int64  
 10  height                        1170 non-null   int64  
 11  weight                        1170 non-null   float64
dtypes: float64(1), int64(5), object(6)
memory usage: 118.8+ KB


<hr>

<hr>

<hr>

<hr>

# 6. Fixing Zip Codes

Dealing with zip codes is often a fraught situation during a data analysis project. The zip codes in your variables `hubs` and `spokes` are at this point numbers. The issue with them being numbers is that for zip codes that begin with a 0, that digit is lost, resulting in a 4-digit instead of 5-digit representation. One approach to correcting this issue is to convert the zip codes to strings and make sure they have the leading zeros as necessary.

1. Using your existing `hubs` variable, convert the zip code to a string ensuring that there are five digits for all zip codes.
2. Using your existing `spokes` variable, convert the zip code to a string ensuring that there are five digits for all zip codes.

In [11]:
hubs['zip'] = hubs['zip'].astype(str).str.zfill(5)

spokes['zip'] = spokes['zip'].astype(str).str.zfill(5)

C:\Users\danie\AppData\Local\Temp\ipykernel_34264\632646081.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hubs['zip'] = hubs['zip'].astype(str).str.zfill(5)


In [12]:
# This is a test cell
# If **NO** message is printed, it means that the tests passed
# One basic test to see if your code works
assert_equal(hubs.zip.dtype, object, msg='You did not fix the hub zip codes correctly')

<hr>

<hr>

# 7. Exploring Requests

You are interested in understanding how the requests from different origin cities vary in terms of the type of freight customers are requesting to be moved. That is, how many different boxes, crates, and pallets are being requested for each origin city. Using your clean `requests` data, create a frequency table where the rows are the origin city and the columns indicate if the request was for a box, crate, or pallet. Store your frequency table in a variable named `oc_type_table`.

In [13]:
oc_type_table = requests.groupby('origin_city')['handling_unit'].value_counts().unstack()

oc_type_table

handling_unit,box,crate,pallet
origin_city,,,
Akron,4.0,18.0,32.0
Allentown,9.0,19.0,37.0
Asheville,NaN,NaN,2.0
Athens,1.0,9.0,12.0
Atlanta,4.0,3.0,20.0
Augusta,1.0,15.0,19.0
Birmingham,NaN,8.0,26.0
Catskill,5.0,5.0,17.0
Charlotte,3.0,9.0,27.0


In [14]:
# This is a test cell
# If **NO** message is printed, it means that the tests passed
# One basic test to see if your code works
assert_equal(oc_type_table.shape[0], 34, msg='Your variable oc_type_table does not have the correct number of rows')
assert_equal(oc_type_table.shape[1], 3, msg='Your variable oc_type_table does not have the correct number of columns')

<hr>

# 8. Exploring Requests Continued

Now you are interested in understanding the flow of requests from originating states to destination states. 

1. Using your clean `requests` data, create a frequency table where the rows are the origin state and the columns are the destination state. Store your frequency table in a variable named `od_table`.
2. Create a `Series` variable named `od_most_requests` that contains the origin state as the index and the values contain the destination state with the most requests along that route.

In [15]:
od_table = pd.crosstab(index=requests.origin_state, columns=requests.destination_state, colnames=['destination_state'])

od_most_requests = od_table.idxmax(axis=1)
od_most_requests

origin_state
AL    PA
GA    VA
MD    OH
NC    PA
NJ    VA
NY    VA
OH    VA
PA    PA
SC    VA
TN    PA
VA    PA
WV    PA
dtype: object

In [16]:
# This is a test cell
# If **NO** message is printed, it means that the tests passed
# One basic test to see if your code works
assert_equal(od_table.shape[0], 12, msg='Your variable od_table does not have the correct number of rows')
assert_equal(od_table.shape[1], 12, msg='Your variable od_table does not have the correct number of columns')

In [17]:
# This is a test cell
# If **NO** message is printed, it means that the tests passed
# One basic test to see if your code works
assert_equal(od_most_requests.index.name, 'origin_state', msg='You did not create the Series `od_most_requests` with the correct index.')

<hr>

**&copy; 2022 - Present: Matthew D. Dean, Ph.D.   
Clinical Associate Professor of Business Analytics at William \& Mary.**